In [1]:
!pip install -U google-genai gradio python-docx reportlab

In [2]:
from google import genai
import os
from getpass import getpass
from docx import Document
from reportlab.lib.pagesizes import LETTER
from reportlab.pdfgen import canvas
import gradio as gr

In [3]:
if "GEMINI_API_KEY" not in os.environ:
    os.environ["GEMINI_API_KEY"] = getpass("Enter your Gemini API Key: ")
client = genai.Client(api_key=os.environ["GEMINI_API_KEY"])


Enter your Gemini API Key: ··········


In [4]:
# Resume Generator
def generate_resume(name, role, experience, skills, education, email, phone, linkedin):
    prompt = f"""
    Create an ATS-friendly resume.

    Candidate Name: {name}
    Target Role: {role}
    Years of Experience: {experience}
    Key Skills: {skills}
    Education:{education}
    Email: {email}
    Phone: {phone}
    LinkedIn: {linkedin}

    Generate the resume with the following sections, elaborating on the provided information:

    - **Summary:** A concise professional summary highlighting key achievements and career goals.
    - **Experience:** Detail relevant work experience, using the provided 'Years of Experience' and 'Key Skills' to infer typical responsibilities and achievements for the 'Target Role'. Use bullet points for accomplishments. Do not use generic placeholders like '[Company Name]'.
    - **Skills:** List and briefly describe proficiency in the 'Key Skills' mentioned.
    - **Education:** Assume typical education for the role if not explicitly provided (e.g., Bachelor's or Master's degree in a relevant field from a reputable university, along with relevant coursework or honors).
    - **Projects:** Assume typical projects for the role if not explicitly provided.

    Ensure the resume is:
    - ATS-friendly.
    - Clean and professional in formatting.
    - Uses bullet points for achievements and responsibilities.
    - Incorporates the provided contact information.
    """

    return generate_text(prompt)

In [5]:
# Cover Letter Generator
def generate_cover_letter(name, role, company, job_description, email, phone, linkedin, city, state):
    prompt = f"""
    Write a professional cover letter.

    Candidate: {name}
    Role: {role}
    Company: {company}
    Contact Information:
    Email: {email}
    Phone: {phone}
    LinkedIn: {linkedin}
    Location: {city}, {state}

    Job Description:
    {job_description}

    Tone: confident, professional, concise.
    """
    return generate_text(prompt)

In [6]:
#DOCX Export
def export_docx(text, filename="resume.docx"):
    doc = Document()
    for line in text.split("\n"):
        doc.add_paragraph(line)
    doc.save(filename)
    return filename

In [7]:
# PDF Export
def export_pdf(text, filename="resume.pdf"):
    c = canvas.Canvas(filename, pagesize=LETTER)
    width, height = LETTER
    y = height - 40

    for line in text.split("\n"):
        c.drawString(40, y, line)
        y -= 14
        if y < 40:
            c.showPage()
            y = height - 40

    c.save()
    return filename

In [8]:
def generate_text(prompt: str) -> str:
    try:
        # client is initialized in mZbboKx1gFBG
        response = client.models.generate_content(
            model="gemini-2.5-flash",
            contents=prompt
        )
        return response.text
    except Exception as e:
        return f"❌ Gemini error: {str(e)}"

In [9]:
# Gradio UI
def app(name, role, experience, skills, email, phone, education, linkedin, city, state, company, job_description):
    # Generate content
    resume_content = generate_resume(
        name, role, experience, skills, email, education, phone, linkedin
    )
    cover_letter_content = generate_cover_letter(
        name, role, company, job_description, email, phone, linkedin, city, state
    )

    # Export resume with specific filenames
    resume_docx_path = export_docx(resume_content, filename=f"Resume_{name.replace(' ', '_')}.docx")
    resume_pdf_path = export_pdf(resume_content, filename=f"Resume_{name.replace(' ', '_')}.pdf")

    # Export cover letter with specific filenames
    cover_letter_docx_path = export_docx(cover_letter_content, filename=f"CoverLetter_{name.replace(' ', '_')}_{company.replace(' ', '_')}.docx")
    cover_letter_pdf_path = export_pdf(cover_letter_content, filename=f"CoverLetter_{name.replace(' ', '_')}_{company.replace(' ', '_')}.pdf")

    # Return text content and paths for download
    return resume_content, cover_letter_content, resume_docx_path, resume_pdf_path, cover_letter_docx_path, cover_letter_pdf_path

interface = gr.Interface(
    fn=app,
    inputs=[
        gr.Textbox(label="Name"),
        gr.Textbox(label="Email"),
        gr.Textbox(label="Phone Number"),
        gr.Textbox(label="LinkedIn URL"),
        gr.Textbox(label="City"),
        gr.Textbox(label="State"),
        gr.Textbox(label="Role"),
        gr.Textbox(label="Education"),
        gr.Textbox(label="Years of Experience"),
        gr.Textbox(label="Skills"),
        gr.Textbox(label="Company Name"),
        gr.Textbox(label="Job Description", lines=6),
    ],
    outputs=[
        gr.Textbox(label="Generated Resume Text", lines=20),
        gr.Textbox(label="Generated Cover Letter Text", lines=15),
        gr.File(label="Download Resume (DOCX)"),
        gr.File(label="Download Resume (PDF)"),
        gr.File(label="Download Cover Letter (DOCX)"),
        gr.File(label="Download Cover Letter (PDF)"),
    ],
    title="AI Resume & Cover Letter Generator"
)

interface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://92fb49192eacca4472.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
